# Cell list

The sampling keeps its computational complexity managable by dividing the box up into cells that are just a bit larger than one particle diameter, which one could achieve, for example, by setting the number of divisions to $\left\lfloor\frac{L}{\sigma}\right\rfloor$ where $L$ is simulation box size and $\sigma$ particle diameter. The reason for this is that particle pairs that are not in the same cell or neighbouring cells are definitely out of their interaction range $\sigma$, making it redundant to compute their potential energy.

For this reason we can, in fact, also safely update cells that are two apart at the same time. Cells will be two apart if we give any given cell of an update group an immediate neighbourhood of cells (as skin, if you will) of thickness one, which will be $3^d$ in count; every part of this representative cell neighbourhood can be thought of as standing for the entire parallel-updating group. The number of divisions will have to divide $3$ for this to fit cleanly, so the number of divisions we choose is $$3\left\lfloor\frac{1}{3}\frac{L}{\sigma}\right\rfloor$$

# Limits to move magnitude

The limit to the size of moves is one particle diameter, since otherwise two particles from the same cell group could potentially interact


### Illustration
<div style="text-align: center;"><img src="cell_list_illustration.png" alt="Illustration of why half a particle diameter is the worst-case upper bound for move size" width="50%"></div>

In [1]:
import numpy as np
import numba as nb

In [2]:
from math import lgamma

@nb.njit
def B(d):
    '''-
    Returns the hyper-volume of the unit ball
    in d dimensions

    https://en.wikipedia.org/wiki/N-sphere#Volume_and_area
    '''
    gamma_of_1_plus_d_over_2 = np.exp(lgamma(1 + d/2))
    return np.pi**(d/2) / gamma_of_1_plus_d_over_2

In [3]:
@nb.njit
def u(r, E, sigma):
	'''
	pair potential for penetrable spheres or disks given a radius
	'''
	return E if r < sigma else (E/2 if r == sigma else 0)

In [ ]:
from random import shuffle



@nb.njit
def __repopulate_cell_list(
    bin_indices, cell_list, cell_list_size,
    particle_number, spatial_dimension_count,
    divisions):
    '''
    takes in the array of positions and fills them
    into the cell list
    '''
    # clearing all lists entries
    for i in range(cell_list_size):
        cell_list[i].clear()
    # bin indices are given by flooring
        
    for k in range(particle_number):
        # flattening the indices ...
        b_k = bin_indices[k]
        # by: b_k[0] + divisions * b_k[1] + ... + divisions**(d-1) * b_k[d-1]
        flattened_index = 0
        for l in range(spatial_dimension_count):
            flattened_index += int(b_k[l] * divisions**l)
        cell_list[flattened_index].append(k)



@nb.njit(parallel=True)
def run(
    overlap_energy, kB_times_temperature,
    packing_density, particle_number, spatial_dimension_count=3,
    simulation_box_size=1, move_count=1000):
    '''
    Executing the metropolis hastings sampling of
    the penetrable sphere system
    '''
    # computing the particle diameter corresponding to the ...
    # ... sought packing density
    volume = simulation_box_size**spatial_dimension_count
    number_density = particle_number / volume
    sigma = 2 * (
        packing_density / (
            B(spatial_dimension_count) * number_density)
            )**(1/spatial_dimension_count)
    # randomly placed particles
    x = np.random.rand(particle_number, spatial_dimension_count)
    # cell list for energy computation, where the number of divisions is ...
    # ... chosen to be multiples of three, in order to update cells later in ...
    # ... 3^d groups (hence the factors of 3 and 1/3 sandwiching the floor-operation)
    third_of_divisions = int(np.floor((1/3)*simulation_box_size / sigma)) # NOTE: used a lot later to inflate flattened indices
    divisions = int(3 * third_of_divisions)
    cell_list_size = divisions**spatial_dimension_count
    cells_per_update_group = int(cell_list_size / 3**spatial_dimension_count)
    # functions need a fixed return type, so we cannot recursively construct ...
    # ... the cell list without some overloading magic. Hence I will just use ...
    # ... a flattened index
    cell_list = nb.typed.List([
        # the number of divisions per dimension
        nb.typed.List.empty_list(nb.types.int64)
        for _ in range(cell_list_size)
    ])
    for _ in range(move_count):
        
        # executing the sampling moves where we update ...
        # ... non-interacting cells in parallel, i.e. one of ...
        # ... 3^d groups of cells which are all out of ...
        # ... interaction range of each other; cells, separated ...
        # ... by three in any direction instead of one, are ...
        # ... updated, i.e. i_k = 3*n_k + g_k, where g=(g_0,...g_{d-1}) is the ...
        # ... index of the group, as represented by a virtual 3x3x...x3 ...
        # ... configuration of cells and n_k is the cell index
        
        # we start at g_0 = g_1 = ... = g_{d-1} = -1 and treat ...
        # ... -1, 0 and 1 as digits. if the previous digit ...
        # ... g_{k-1} = 2, it has overflow and g_k is increased
        g = np.zeros(spatial_dimension_count, dtype=np.int64)
        
        while True:
            # we want the cell list to reflect previosly done moves before ...
            # ... updating the next group
            bin_indices = np.floor(divisions * x / simulation_box_size) % divisions
            __repopulate_cell_list(
                bin_indices, cell_list, cell_list_size,
                particle_number, spatial_dimension_count,
                divisions
            )
            # we now update all cells of group g in parallel
            for intra_group_index in nb.prange(cells_per_update_group):
                # we need to inflate the index, for which we want to modify it; hence ...
                # ... we make a copy to mess with
                igi = int(intra_group_index)
                # finally, check if we even want to increment the ...
                # ... cell multi-index once more ...
                cell_index = 0
                for k in range(spatial_dimension_count):
                    # intra_group_index = n_0 + n_1 * M + n_2 * M**2 + ... + n_{d-1} * M**(d-1), hence ...
                    # ... intra_group_index % M = n_0, making intra_group_index - n_0 = n_1 * M + ..., such ...
                    # ... that way may divide out M and repeat the process. M here is the numer of cells ...
                    # ... per update group
                    n_k = igi % third_of_divisions
                    cell_index += int((3 * n_k + g[k]) * divisions**k)
                    igi = igi - n_k
                    igi = int(igi / third_of_divisions)
                # for all particles in the cell in question, we will now
                u(0, overlap_energy, sigma) # TODO PER PARTICLE

            # finally, check if we even want to increment the group again ...
            if np.sum(g) == spatial_dimension_count * 2:
                break
            # ... but if we do, want to, increment the first digit ...
            g[0] += 1
            # ... and update the rest from lowest to highest
            for k in range(1, spatial_dimension_count):
                if g[k-1] > 2: # two is the highest digit
                    g[k-1] = 0
                    g[k] += 1

        # shuffling all particles to not accidentally introduce ...
        # ... effects only due to the fixed update order
        shuffle(x)


In [10]:
run(
    1, # overlap energy
    1, # kBT
    0.3, # packing density
    1000, # particle number
    move_count=1000
)

#### Non-Parallelized

compile time 1: `22.4s`

time 1: `11.1s`

#### Parallelized

compile time 2: `25.8`

time 2: `15.3s`